## Markdown Scrape & Export (Files Only)

Fetch markdown files from GitHub (`uvarc/rc-learning`) and save them as local `.md` files.

This trims `scrape-md-new.ipynb` down to only the fetch + export steps, dropping the DB/vector-store ingestion — matching the pattern used in `scrape-jira-new.ipynb`'s markdown-only variant.

## 1. Fetch Markdown from GitHub

Uses the GitHub tree API to list all `.md` files under `content/`, then fetches each via raw URL.
Skips drafts (`draft = true` / `draft: true` in frontmatter).

In [ ]:
import requests
from langchain_core.documents import Document

GITHUB_REPO = "uvarc/rc-learning"
GITHUB_BRANCH = "main"
CONTENT_PATH = "content"

def load_documents_from_github(repo=GITHUB_REPO, branch=GITHUB_BRANCH, content_path=CONTENT_PATH):
    tree_url = f"https://api.github.com/repos/{repo}/git/trees/{branch}?recursive=1"
    resp = requests.get(tree_url, timeout=30)
    resp.raise_for_status()
    tree = resp.json().get("tree", [])

    md_files = [
        item for item in tree
        if item["type"] == "blob"
        and item["path"].startswith(content_path + "/")
        and item["path"].endswith(".md")
    ]
    print(f"Found {len(md_files)} markdown files in {repo}/{content_path}")

    documents = []
    for item in md_files:
        raw_url = f"https://raw.githubusercontent.com/{repo}/{branch}/{item['path']}"
        r = requests.get(raw_url, timeout=15)
        if r.status_code != 200:
            print(f"Failed to fetch {item['path']}: {r.status_code}")
            continue
        text = r.text

        # Skip TOML-frontmatter drafts (+++ ... +++)
        if text.startswith('+++'):
            end = text.find('+++', 3)
            if end != -1 and 'draft = true' in text[3:end]:
                print(f"Skipping draft: {item['path']}")
                continue

        # Skip YAML-frontmatter drafts (--- ... ---)
        if text.startswith('---'):
            end = text.find('---', 3)
            if end != -1 and 'draft: true' in text[3:end]:
                print(f"Skipping draft: {item['path']}")
                continue

        doc = Document(
            page_content=text,
            metadata={
                "source_type": "markdown",
                "source": item["path"],
                "chunk_number": 0,
            }
        )
        documents.append(doc)

    # Deduplicate by content
    unique = {doc.page_content: doc for doc in documents}
    documents = list(unique.values())
    print(f"Loaded {len(documents)} documents (after dedup)")
    return documents

markdown_documents = load_documents_from_github()

## 2. Preview

In [ ]:
for doc in markdown_documents[:3]:
    print(f"=== {doc.metadata['source']} ===")
    print(doc.page_content[:300])
    print()

## 3. Generate Markdown Files

In [ ]:
from pathlib import Path
import re

PROJECT_ROOT = Path.cwd().parent

# Store generated files inside this repo
OUTPUT_FOLDER = PROJECT_ROOT / "data" / "markdown"

# Automatically create data/markdown if it doesn't exist
OUTPUT_FOLDER.mkdir(
    parents=True,
    exist_ok=True
)


def safe_filename(filename):

    filename = re.sub(
        r'[<>:"\\|?*]',
        '_',
        filename
    )

    filename = re.sub(
        r'\s+',
        ' ',
        filename
    ).strip()

    return filename[:150]


print(
    f"Documents available: "
    f"{len(markdown_documents)}"
)

print(
    f"Writing files to: "
    f"{OUTPUT_FOLDER}"
)


created = 0
failed = 0


for doc in markdown_documents:

    try:

        # e.g. "content/authors/abd/_index.md" -> "authors_abd__index.md"
        relative_path = doc.metadata["source"].split("content/", 1)[-1]

        flat_name = safe_filename(
            relative_path.replace("/", "_")
        )

        file_path = OUTPUT_FOLDER / flat_name

        with open(
            file_path,
            "w",
            encoding="utf-8"
        ) as f:

            f.write(
                doc.page_content.strip() + "\n"
            )

        created += 1

        print(
            f"Created: {file_path.name}"
        )

    except Exception as e:

        failed += 1

        print(
            f"ERROR processing "
            f"{doc.metadata.get('source', 'unknown')}: "
            f"{e}"
        )


print("=" * 60)
print("MARKDOWN GENERATION COMPLETE")
print("=" * 60)

print(f"Created: {created} files")
print(f"Failed:  {failed} files")
print(f"Output folder: {OUTPUT_FOLDER}")